# 21 — Vision Transformer : exploration architecture avant investissement

**Auteur :** Yann Smatti  
**Objectif :** comprendre et tester la faisabilite d'une architecture ViT minimale avant de lui consacrer un vrai protocole comparatif

Ce notebook a une fonction precise : verifier si un ViT merite d'etre pousse plus loin dans ce projet. A ce stade, il s'agit d'un bac a sable technique pour comprendre les patchs, l'encodage positionnel et les blocs d'attention, pas d'un candidat direct a la production.

## Pourquoi le garder

- il permet de documenter une piste modele distincte des CNN ;
- il sert de support pour discuter capacite, regularisation et besoins en donnees ;
- il aide a eviter d'introduire un ViT sans base comparative ni justification empirique.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMAGE_SIZE = 224
PATCH_SIZE = 16
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2
PROJECTION_DIM = 64
NUM_HEADS = 4
TRANSFORMER_UNITS = [128, 64]
TRANSFORMER_LAYERS = 4
MLP_HEAD_UNITS = [128, 64]
NUM_CLASSES = 2


In [ ]:
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )
        patch_dims = patches.shape[-1]
        return tf.reshape(patches, [batch_size, -1, patch_dims])

class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patch):
        positions = tf.range(start=0, limit=NUM_PATCHES, delta=1)
        return self.projection(patch) + self.position_embedding(positions)


In [ ]:
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x


def create_vit_classifier():
    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    patches = Patches(PATCH_SIZE)(inputs)
    encoded_patches = PatchEncoder(NUM_PATCHES, PROJECTION_DIM)(patches)

    for _ in range(TRANSFORMER_LAYERS):
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=PROJECTION_DIM, dropout=0.1)(x1, x1)
        x2 = layers.Add()([attention_output, encoded_patches])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=TRANSFORMER_UNITS, dropout_rate=0.1)
        encoded_patches = layers.Add()([x3, x2])

    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.3)(representation)
    features = mlp(representation, hidden_units=MLP_HEAD_UNITS, dropout_rate=0.3)
    logits = layers.Dense(NUM_CLASSES, activation='softmax')(features)
    return keras.Model(inputs=inputs, outputs=logits)

vit = create_vit_classifier()
vit.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
vit.summary()


## Suite logique avant toute comparaison serieuse

Pour transformer ce notebook en vraie experimentation, il faudra :

1. raccorder le modele a un dataset versionne et a un protocole de split propre ;
2. comparer a une baseline CNN / EfficientNet / ConvNeXt dans les memes conditions ;
3. suivre la stabilite d'entrainement, la regularisation et le cout compute ;
4. ne retenir cette piste que si elle apporte un gain mesurable ou une meilleure generalisation.

Sans cela, ce notebook doit rester classe comme exploration technique et non comme candidat pre-production.